In [0]:
from pyspark.sql import functions as F

silver = spark.table("silver_cdrs")

print(silver.count())
display(silver.groupBy("status").count())

948


status,count
ANSWERED,718
FAILED,126
BUSY,104


In [0]:
is_answered = F.col("status") == "ANSWERED"

gold_caller = (
    silver
    .groupBy("caller")
    .agg(
        F.count("*").alias("attempts"),
        F.count(F.when(is_answered, 1)).alias("answered"),
        F.sum(F.when(is_answered, F.col("duration_s"))).alias("answered_seconds"),
        F.avg(F.when(is_answered, F.col("duration_s"))).alias("acd_s"),
    )
    .withColumn("asr_pct", F.round(F.col("answered") / F.col("attempts") * 100, 2))
    .withColumn("acd_s", F.round("acd_s", 1))
    .orderBy("caller")
)

display(gold_caller)

caller,attempts,answered,answered_seconds,acd_s,asr_pct
+48123451000,34,24,23935,997.3,70.59
+48123451001,30,22,18809,855.0,73.33
+48123451002,45,31,29958,966.4,68.89
+48123451003,38,27,22793,844.2,71.05
+48123451004,39,26,28097,1080.7,66.67
+48123451005,34,26,26494,1019.0,76.47
+48123451006,37,27,24096,892.4,72.97
+48123451007,43,37,40038,1082.1,86.05
+48123451008,35,24,20231,843.0,68.57
+48123451009,40,33,28452,862.2,82.5


In [0]:
display(gold_caller.agg(F.sum("attempts"), F.sum("answered")))

display(
    gold_caller
    .orderBy(F.col("asr_pct").desc())
)

sum(attempts),sum(answered)
948,718


caller,attempts,answered,answered_seconds,acd_s,asr_pct
+48123451021,42,38,34861,917.4,90.48
+48123451022,43,37,29540,798.4,86.05
+48123451007,43,37,40038,1082.1,86.05
+48123451014,53,44,44218,1005.0,83.02
+48123451010,35,29,27451,946.6,82.86
+48123451009,40,33,28452,862.2,82.5
+48123451023,34,28,25908,925.3,82.35
+48123451020,43,35,29013,828.9,81.4
+48123451024,48,37,30046,812.1,77.08
+48123451005,34,26,26494,1019.0,76.47


ASR (Answer-Seizure Ratio) measures the percentage of call attempts that were successfully answered, indicating the rate of reaching the B-party.

ASR = answered / attempts × 100

ACD (Average Call Duration) measures the average duration of answered calls. It helps indicate how long successful calls remain and show potential issues if there's a cut-off pattern at a certain duration.

ACD = answered seconds / answered calls

In [0]:
gold_caller.write.mode("overwrite").saveAsTable("gold_caller_kpis")

totals = spark.table("gold_caller_kpis").agg(
    F.sum("attempts").alias("attempts"),
    F.sum("answered").alias("answered"),
).first()

silver_n        = silver.count()
silver_answered = silver.filter(is_answered).count()

print(f"gold attempts={totals['attempts']}  silver rows={silver_n}")
print(f"gold answered={totals['answered']}  silver answered={silver_answered}")

assert totals["attempts"] == silver_n, "Gold attempts don't match silver row count"
assert totals["answered"] == silver_answered, "Gold answered doesn't match silver"
print("Gold caller KPIs OK")

gold attempts=948  silver rows=948
gold answered=718  silver answered=718
Gold caller KPIs OK


In [0]:
gold_hourly = (
    silver
    .withColumn("hour", F.date_trunc("hour", F.col("start_ts")))
    .groupBy("hour")
    .agg(
        F.count("*").alias("attempts"),
        F.count(F.when(is_answered, 1)).alias("answered"),
        F.avg(F.when(is_answered, F.col("duration_s"))).alias("acd_s"),
    )
    .withColumn("asr_pct", F.round(F.col("answered") / F.col("attempts") * 100, 2))
    .withColumn("acd_s", F.round("acd_s", 1))
    .orderBy("hour")
)

gold_hourly.write.mode("overwrite").saveAsTable("gold_hourly_kpis")
display(gold_hourly)

hour,attempts,answered,acd_s,asr_pct
2025-12-02T11:00:00.000Z,1,1,1726.0,100.0
2025-12-02T12:00:00.000Z,2,1,627.0,50.0
2025-12-02T13:00:00.000Z,3,3,790.3,100.0
2025-12-02T14:00:00.000Z,1,1,222.0,100.0
2025-12-02T15:00:00.000Z,2,1,1344.0,50.0
2025-12-02T17:00:00.000Z,2,1,26.0,50.0
2025-12-02T18:00:00.000Z,3,3,718.3,100.0
2025-12-02T19:00:00.000Z,1,0,null,0.0
2025-12-02T20:00:00.000Z,4,3,967.7,75.0
2025-12-02T21:00:00.000Z,1,0,null,0.0


In [0]:
totals = spark.table("gold_hourly_kpis").agg(
    F.sum("attempts").alias("attempts"),
    F.sum("answered").alias("answered"),
).first()

silver_n        = silver.count()
silver_answered = silver.filter(is_answered).count()

print(f"gold attempts={totals['attempts']}  silver rows={silver_n}")
print(f"gold answered={totals['answered']}  silver answered={silver_answered}")

assert totals["attempts"] == silver_n, "Gold attempts don't match silver row count"
assert totals["answered"] == silver_answered, "Gold answered doesn't match silver"
print("Gold hourly KPIs OK")

gold attempts=948  silver rows=948
gold answered=718  silver answered=718
Gold hourly KPIs OK


With such a small dataset, most hours contain only 1–2 calls, so hourly ASR varies between 0%, 50% and 100% therefore it isn't very meaningful. In production either the window would've been widened, or only samples with a certain minimum number of attempts would've been looked at.